# 06.2 · Engenharia de features — cestas para o GMM
**Projeto:** Estágio Engenharia de Dados · **Squad:** 3 — Batch · **Autor:** Alexandro Silva

Prepara os dados da Silver `physical_itens_venda_caixa` para o modelo do problema **P2 · Segmentação de
cestas de compra** (planilha de ML da Squad 3), modelo **B: Gaussian Mixture (GMM)**. Transforma itens em
**uma linha por cesta (`id_transacao`)** com colunas numéricas que ajudam a identificar padrões de compra.

## Regras de transformação
| Nº | Regra | Parâmetro |
|---|---|---|
| R1 | Cada item recebe a `categoria_pai` do catálogo (e-commerce via `ecommerce_categorias`, perecíveis direto) | — |
| R2 | Itens sem categoria vão para `outras` | `TRATAR_SEM_CATEGORIA` |
| R3 | Categorias raras são **mantidas** (sem agrupamento) | `LIMIAR_CATEGORIA_RARA = 0` |
| R4 | Cestas com flag: **excluídas se forem poucas** (< 1% das cestas), mantidas se forem muitas | `LIMIAR_EXCLUSAO_FLAG` |
| R5 | `n_itens` e `valor_total` recebem `log1p` para reduzir a assimetria | — |
| R6 | Colinearidade das `pct_*` (somam 1): retira-se uma categoria de referência | `COLINEARIDADE` |
| R7 | Cestas de 1 item são **tratadas à parte**: ficam na tabela, marcadas, e não entram no GMM | `TRATAR_ITEM_UNICO_A_PARTE` |

## Features
| Grupo | Colunas | Entra no modelo? |
|---|---|---|
| Planilha de ML | `n_itens`, `valor_total`, `pct_<categoria>` | ✅ (transformadas: `log_n_itens`, `log_valor_total`, mix sem a referência) |
| Composição da cesta | `n_skus`, `n_categorias`, `pct_itens_pesaveis`, `hhi_mix` | ➖ só análise (decidir com a dupla 2) |
| Preço | `preco_medio_item`, `log_preco_medio_item` | ➖ só análise |
| Controle | `tem_flag_valor`, `tem_flag_fracionada`, `tem_sem_categoria` | ❌ |
| Segmentação | `cesta_item_unico`, `entra_modelo` | ❌ — o notebook de ML filtra `entra_modelo = 1` |

> A **padronização** (`StandardScaler`) não é feita aqui: ela é ajustada no notebook de ML, só com a base de
> treino, para não vazar informação.

## Origem e destino
| Item | Definição |
|---|---|
| Silver de itens | `abfss://squad3@internshipdatalake.dfs.core.windows.net/silver/physical_itens_venda_caixa` |
| Catálogo | **provisório:** `workspace.squad3.ecommerce_produtos` + `ecommerce_categorias` + `physical_produtos_pereciveis` |
| Destino | `abfss://squad3@internshipdatalake.dfs.core.windows.net/features/cestas_gmm` (Delta, partição `ano`/`mes`) |
| Metadados | `.../features/cestas_gmm_metadados` (parâmetros, categorias e lista de features da execução) |

## Etapas
1. Credenciais e caminhos
2. Parâmetros
3. Leitura da Silver e do catálogo
4. Categoria por item (R1–R3)
5. Agregação por cesta e filtros (R4)
6. Mix de categorias (`pct_*`)
7. Features extras e transformação log (R5)
8. Tratamento da colinearidade (R6)
9. Validação das features
10. Gravação e conferência

## 1. Credenciais e caminhos
Mesmo padrão dos notebooks 06 e 06.1.

In [0]:
# Célula 1 — Credenciais e caminhos
from dotenv import load_dotenv
import os

load_dotenv("/Workspace/Users/alexandrofs31@gmail.com/.env")

CLIENT_ID       = os.getenv("ADLS_CLIENT_ID")
TENANT_ID       = os.getenv("ADLS_TENANT_ID")
CLIENT_SECRET   = os.getenv("ADLS_CLIENT_SECRET")
STORAGE_ACCOUNT = os.getenv("ADLS_STORAGE_ACCOUNT", "internshipdatalake")

faltando = [n for n, v in {"ADLS_CLIENT_ID": CLIENT_ID, "ADLS_TENANT_ID": TENANT_ID,
                           "ADLS_CLIENT_SECRET": CLIENT_SECRET}.items() if not v]
if faltando:
    raise ValueError(f"Variáveis ausentes no .env: {faltando}")

TABELA            = "physical_itens_venda_caixa"
CONTAINER_SQUAD   = "squad3"
BASE_SQUAD        = f"abfss://{CONTAINER_SQUAD}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
CAMINHO_SILVER    = f"{BASE_SQUAD}/silver/{TABELA}"
CAMINHO_FEATURES  = f"{BASE_SQUAD}/features/cestas_gmm"
CAMINHO_METADADOS = f"{BASE_SQUAD}/features/cestas_gmm_metadados"

# PROVISÓRIO: trocar pela Silver das duplas responsáveis quando forem publicadas
TABELA_PRODUTOS_REF   = "workspace.squad3.ecommerce_produtos"
TABELA_CATEGORIAS_REF = "workspace.squad3.ecommerce_categorias"
TABELA_PERECIVEIS_REF = "workspace.squad3.physical_produtos_pereciveis"

sufixo = f"{STORAGE_ACCOUNT}.dfs.core.windows.net"
opcoes_adls = {
    f"fs.azure.account.auth.type.{sufixo}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{sufixo}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{sufixo}": CLIENT_ID,
    f"fs.azure.account.oauth2.client.secret.{sufixo}": CLIENT_SECRET,
    f"fs.azure.account.oauth2.client.endpoint.{sufixo}": f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token",
}

print(f"📦 Silver   : {CAMINHO_SILVER}")
print(f"🎯 Destino  : {CAMINHO_FEATURES}")

## 2. Parâmetros
Todos os tratamentos ficam aqui, com valores sugeridos. Depois de rodar a EDA 06.1 com os dados reais,
ajuste o que for preciso e rode de novo: o notebook sobrescreve a tabela de features.

| Parâmetro | Padrão | Opções |
|---|---|---|
| `TRATAR_SEM_CATEGORIA` | `"outras"` | `"outras"` · `"excluir_cestas"` (tira a cesta inteira) |
| `LIMIAR_CATEGORIA_RARA` | `0` | `0` = mantém todas; ex.: `0.05` agrupa em `outras` as que aparecem em < 5% das cestas |
| `LIMIAR_EXCLUSAO_FLAG` | `0.01` | cada flag: se atinge **menos** que essa fração das cestas, as cestas com ela são excluídas; se atinge mais, são mantidas |
| `TRATAR_ITEM_UNICO_A_PARTE` | `True` | `True` = cestas de 1 item ficam com `entra_modelo = 0`; `False` = entram no GMM |
| `COLINEARIDADE` | `"remover_categoria"` | `"clr"` · `"clr_pca"` |
| `CATEGORIA_REFERENCIA` | `None` | `None` = `outras` se existir; senão, a de menor receita |

In [0]:
# Célula 2 — Parâmetros
# Decisões validadas por Alex após a EDA 06.1 (nov/2026)
TRATAR_SEM_CATEGORIA      = "outras"   # EDA: 0 itens sem categoria
LIMIAR_CATEGORIA_RARA     = 0          # manter todas as categorias
LIMIAR_EXCLUSAO_FLAG      = 0.01       # flag em < 1% das cestas → exclui; ≥ 1% → mantém (EDA: 0,00%)
TRATAR_ITEM_UNICO_A_PARTE = True       # EDA: 2,67% das cestas têm 1 item

COLINEARIDADE        = "remover_categoria"   # "remover_categoria" | "clr" | "clr_pca"
CATEGORIA_REFERENCIA = None                  # nome da categoria_pai (minúsculo) ou None
PSEUDOCONTAGEM_CLR   = 1e-3                  # evita log(0) no CLR
VARIANCIA_PCA        = 0.90                  # variância acumulada mantida no PCA

UNIDADES_PESAVEIS = ["kg", "g", "l", "ml"]
AMOSTRA_MAX = 200_000   # amostra levada ao pandas para validação e para ajustar o PCA
SEED = 42
GRAVAR = True

assert TRATAR_SEM_CATEGORIA in ("outras", "excluir_cestas")
assert COLINEARIDADE in ("remover_categoria", "clr", "clr_pca")
print("⚙️ Parâmetros:", {k: v for k, v in globals().items()
                        if k.isupper() and k not in ("CLIENT_ID", "TENANT_ID", "CLIENT_SECRET") and not k.startswith(("CAMINHO", "TABELA", "BASE"))})

## 3. Leitura da Silver e do catálogo
Mesma lógica da EDA 06.1:
- **E-commerce:** o produto aponta para uma subcategoria (`id_categoria`), e a categoria pai vem de
  `ecommerce_categorias`. Se o produto já estiver numa categoria de topo, vale o próprio `nome_categoria`.
- **Perecíveis:** já trazem `categoria_pai`.
- SKU presente nos dois catálogos fica com a categoria do e-commerce.

In [0]:
# Célula 3 — Leitura da Silver e do catálogo unificado
from functools import reduce
from operator import add
import unicodedata, json
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.max_columns", 60)

df = spark.read.format("delta").options(**opcoes_adls).load(CAMINHO_SILVER)
total_itens = df.count()
print(f"✅ Silver lida: {total_itens:,} itens")

df_categorias = spark.table(TABELA_CATEGORIAS_REF).select(
    F.col("id_categoria").cast("int").alias("id_categoria"),
    F.coalesce(F.col("nome_categoria_pai"), F.col("nome_categoria")).alias("nome_pai"))

cat_ecommerce = (spark.table(TABELA_PRODUTOS_REF)
    .withColumn("id_categoria", F.col("id_categoria").cast("int"))
    .join(df_categorias, on="id_categoria", how="left")
    .select(F.lower(F.trim(F.col("sku"))).alias("codigo_barras_produto"),
            F.lower(F.trim(F.col("nome_pai"))).alias("categoria_pai"),
            F.lit(1).alias("prioridade")))

cat_pereciveis = (spark.table(TABELA_PERECIVEIS_REF)
    .select(F.lower(F.trim(F.col("sku"))).alias("codigo_barras_produto"),
            F.lower(F.trim(F.col("categoria_pai"))).alias("categoria_pai"),
            F.lit(2).alias("prioridade")))

df_catalogo = (cat_ecommerce.unionByName(cat_pereciveis)
    .filter(F.col("categoria_pai").isNotNull())
    .withColumn("_rn", F.row_number().over(Window.partitionBy("codigo_barras_produto").orderBy("prioridade")))
    .filter("_rn = 1").select("codigo_barras_produto", "categoria_pai"))

print(f"📚 SKUs no catálogo com categoria_pai: {df_catalogo.count():,}")

## 4. Categoria por item (R1–R3)
Cada item recebe a `categoria_pai` e, a partir dela, a **`categoria_modelo`**, que é a usada nas features:
- itens sem categoria → `outras` (ou a cesta é excluída na etapa 5, conforme `TRATAR_SEM_CATEGORIA`);
- categorias presentes em menos de `LIMIAR_CATEGORIA_RARA` das cestas → `outras` (com `0`, nenhuma é agrupada).

**Decisão:** manter todas as 18 categorias. A tabela abaixo mostra o % de cestas de cada uma; se o GMM
ficar instável por causa de colunas quase sempre zero, basta subir o limiar e rodar de novo.

In [0]:
# Célula 4 — categoria_pai e categoria_modelo
df_enr = (df.withColumn("codigo_barras_produto", F.lower(F.trim("codigo_barras_produto")))
            .join(F.broadcast(df_catalogo), on="codigo_barras_produto", how="left")
            .withColumn("sem_categoria", F.col("categoria_pai").isNull())
            .withColumn("categoria_pai", F.coalesce("categoria_pai", F.lit("sem_categoria"))))

total_transacoes = df_enr.select("id_transacao").distinct().count()

cobertura = (df_enr.groupBy("categoria_pai")
    .agg(F.countDistinct("id_transacao").alias("cestas"),
         F.sum("valor_total_item").cast("double").alias("receita"))
    .toPandas())
cobertura["% cestas"] = cobertura.cestas / total_transacoes
cobertura["% receita"] = cobertura.receita / cobertura.receita.sum()
cobertura["rara"] = cobertura["% cestas"] < LIMIAR_CATEGORIA_RARA
cobertura["categoria_modelo"] = np.where(
    cobertura.rara | (cobertura.categoria_pai == "sem_categoria"), "outras", cobertura.categoria_pai)
cobertura = cobertura.sort_values("receita", ascending=False).reset_index(drop=True)

mapa = dict(zip(cobertura.categoria_pai, cobertura.categoria_modelo))
expr_mapa = F.create_map(*[F.lit(x) for kv in mapa.items() for x in kv])
df_enr = df_enr.withColumn("categoria_modelo", expr_mapa[F.col("categoria_pai")])

agrupadas = cobertura.loc[cobertura.categoria_modelo == "outras", "categoria_pai"].tolist()
print(f"🧺 Transações: {total_transacoes:,}")
print(f"🏷️ categoria_pai: {len(cobertura)} | categoria_modelo: {cobertura.categoria_modelo.nunique()}")
print(f"   Agrupadas em 'outras': {agrupadas or 'nenhuma'}")
display(cobertura)

## 5. Agregação por cesta, filtros (R4) e segmentação (R7)
Uma linha por `id_transacao`. Além das features da planilha, calcula colunas de **composição** e de
**controle**, usadas nos filtros e na análise dos clusters.

**R4 — flags (regra automática):** para cada flag, mede o % de cestas atingidas. Se for **pequeno**
(abaixo de `LIMIAR_EXCLUSAO_FLAG`), excluir custa pouco e as cestas saem. Se for **grande**, excluir
tiraria muita base, e elas ficam (a coluna `tem_flag_*` permite filtrar depois). A regra é recalculada
a cada carga.

**R7 — cestas de 1 item:** não são excluídas. Ficam na tabela com `cesta_item_unico = 1` e
`entra_modelo = 0`, e formam um segmento próprio, definido por regra ("compra de item único").
O GMM é treinado só com `entra_modelo = 1`.

| Coluna | Definição |
|---|---|
| `n_itens` | nº de linhas de item |
| `n_skus` | nº de SKUs distintos |
| `n_categorias` | nº de `categoria_pai` distintas (antes do agrupamento) |
| `valor_total` | soma de `valor_total_item` |
| `pct_itens_pesaveis` | fração dos itens vendidos por peso/volume (kg, g, l, ml) |
| `tem_flag_valor` / `tem_flag_fracionada` / `tem_sem_categoria` | 1 se algum item da cesta tem a situação |
| `cesta_item_unico` | 1 se `n_itens = 1` |
| `entra_modelo` | 1 se a cesta vai para o GMM |

In [0]:
# Célula 5 — Agregação por cesta e filtros
base_cesta = (df_enr.groupBy("id_transacao")
    .agg(F.min("ano").alias("ano"), F.min("mes").alias("mes"),
         F.count("*").alias("n_itens"),
         F.countDistinct("codigo_barras_produto").alias("n_skus"),
         F.countDistinct("categoria_pai").alias("n_categorias"),
         F.sum("valor_total_item").cast("double").alias("valor_total"),
         F.avg(F.lower(F.col("unidade_medida")).isin(UNIDADES_PESAVEIS).cast("int")).alias("pct_itens_pesaveis"),
         F.max(F.col("flag_valor_inconsistente").cast("int")).alias("tem_flag_valor"),
         F.max(F.col("flag_quantidade_fracionada_un").cast("int")).alias("tem_flag_fracionada"),
         F.max(F.col("sem_categoria").cast("int")).alias("tem_sem_categoria"))
    .fillna(0, subset=["pct_itens_pesaveis"]))

base_cesta = base_cesta.cache()
antes = base_cesta.count()

# R4 — regra automática das flags
decisao_flags = []
filtros = [("valor_total > 0", F.col("valor_total") > 0)]
for col_flag, nome_flag in [("tem_flag_valor", "flag_valor_inconsistente"),
                            ("tem_flag_fracionada", "flag_quantidade_fracionada_un")]:
    n_flag = base_cesta.filter(F.col(col_flag) == 1).count()
    pct_flag = n_flag / antes
    acao = "excluir" if pct_flag < LIMIAR_EXCLUSAO_FLAG else "manter"
    decisao_flags.append({"flag": nome_flag, "cestas com a flag": n_flag, "% cestas": pct_flag * 100, "ação": acao})
    if acao == "excluir":
        filtros.append((f"sem {nome_flag}", F.col(col_flag) == 0))
print(f"🚩 Regra das flags (limiar {LIMIAR_EXCLUSAO_FLAG:.0%} das cestas):")
display(pd.DataFrame(decisao_flags))

if TRATAR_SEM_CATEGORIA == "excluir_cestas":
    filtros.append(("sem item sem categoria", F.col("tem_sem_categoria") == 0))

log_filtros = []
for nome, cond in filtros:
    removidas = base_cesta.filter(~cond).count()
    log_filtros.append({"filtro": nome, "cestas removidas": removidas, "% do total": removidas / antes * 100})
    base_cesta = base_cesta.filter(cond)
# R7 — cestas de 1 item: segmento à parte (ficam na tabela, fora do modelo)
base_cesta = (base_cesta
    .withColumn("cesta_item_unico", (F.col("n_itens") == 1).cast("int"))
    .withColumn("entra_modelo", F.lit(1) - F.col("cesta_item_unico") if TRATAR_ITEM_UNICO_A_PARTE else F.lit(1)))

total_cestas = base_cesta.count()
total_modelo = base_cesta.filter("entra_modelo = 1").count()

print(f"\n🧺 Cestas: {antes:,} → {total_cestas:,} após filtros ({(antes - total_cestas) / antes:.2%} removidas)")
display(pd.DataFrame(log_filtros))
print(f"🎯 Entram no GMM : {total_modelo:,} ({total_modelo / total_cestas:.2%})")
print(f"🛒 Item único    : {total_cestas - total_modelo:,} ({(total_cestas - total_modelo) / total_cestas:.2%}) → segmento à parte")

## 6. Mix de categorias (`pct_*`)
Para cada `categoria_modelo`: receita da categoria na cesta ÷ `valor_total` da cesta. Os nomes das colunas
são normalizados (sem acento, espaço ou símbolo). A célula confere que as `pct_*` somam 1 em toda cesta.

In [0]:
# Célula 6 — Pivot de receita por categoria_modelo
def nome_coluna(c):
    s = unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode()
    s = "".join(ch if ch.isalnum() else "_" for ch in s.lower())
    return "_".join(p for p in s.split("_") if p)

categorias = sorted(cobertura.categoria_modelo.unique())
COLS_PCT = [f"pct_{nome_coluna(c)}" for c in categorias]
assert len(set(COLS_PCT)) == len(COLS_PCT), f"Nomes de coluna repetidos após normalização: {COLS_PCT}"

receita_cat = (df_enr.groupBy("id_transacao").pivot("categoria_modelo", categorias)
               .agg(F.sum(F.col("valor_total_item").cast("double"))).fillna(0.0))

cesta = base_cesta.join(receita_cat, on="id_transacao", how="inner")
for c, col_pct in zip(categorias, COLS_PCT):
    cesta = cesta.withColumn(col_pct, F.col(f"`{c}`") / F.col("valor_total"))
cesta = cesta.drop(*categorias)

soma = cesta.select(reduce(add, [F.col(c) for c in COLS_PCT]).alias("s")).agg(F.min("s"), F.max("s")).collect()[0]
print(f"📊 Features de mix: {len(COLS_PCT)} → {COLS_PCT}")
print(f"   Soma das pct_* por cesta: mín {soma[0]:.6f} | máx {soma[1]:.6f} "
      f"{'✅ (somam 1 → colinearidade a tratar na etapa 8)' if abs(soma[0] - 1) < 1e-6 and abs(soma[1] - 1) < 1e-6 else '⚠️ revisar'}")

## 7. Features extras e transformação log (R5)
| Coluna | Definição |
|---|---|
| `log_n_itens` | `log1p(n_itens)` — **entra no modelo** |
| `log_valor_total` | `log1p(valor_total)` — **entra no modelo** |
| `hhi_mix` | soma dos quadrados das `pct_*`: perto de 1 = cesta concentrada numa categoria; perto de 1/k = mix espalhado |
| `preco_medio_item` / `log_preco_medio_item` | `valor_total ÷ n_itens` e seu `log1p` |

`log1p` é usado porque `n_itens` e `valor_total` têm cauda longa à direita (muitas cestas pequenas e
poucas muito grandes), e o GMM supõe grupos com forma normal.

**Resultado da EDA 06.1:** assimetria de `n_itens` cai de 1,94 para 0,08 e a de `valor_total` de 2,31
para −0,54 com o `log1p` (ideal: entre −1 e 1). A padronização (`StandardScaler`) fica no notebook de ML,
ajustada só na amostra de treino.

In [0]:
# Célula 7 — Extras e log
cesta = (cesta
    .withColumn("log_n_itens", F.log1p("n_itens"))
    .withColumn("log_valor_total", F.log1p("valor_total"))
    .withColumn("hhi_mix", reduce(add, [F.col(c) * F.col(c) for c in COLS_PCT]))
    .withColumn("preco_medio_item", F.col("valor_total") / F.col("n_itens"))
    .withColumn("log_preco_medio_item", F.log1p("preco_medio_item")))

assim = cesta.agg(*[F.skewness(c).alias(c) for c in
                    ["n_itens", "log_n_itens", "valor_total", "log_valor_total"]]).collect()[0].asDict()
print("📐 Assimetria antes → depois do log1p:")
print(f"   n_itens     : {assim['n_itens']:7.3f} → {assim['log_n_itens']:7.3f}")
print(f"   valor_total : {assim['valor_total']:7.3f} → {assim['log_valor_total']:7.3f}")

## 8. Tratamento da colinearidade (R6)
As `pct_*` somam 1, então uma delas é sempre `1 − (as outras)`. Com covariância `full`, o GMM precisa
inverter a matriz de covariância, e com colinearidade perfeita ela é **singular**. Três opções:

| `COLINEARIDADE` | O que faz | Colunas de mix no modelo |
|---|---|---|
| `remover_categoria` (padrão) | retira a `pct_` da categoria de referência; a informação continua implícita | `pct_*` sem a referência |
| `clr` | `log(p + ε) − média dos logs da cesta`; depois retira a referência (o CLR soma zero) | `clr_*` sem a referência |
| `clr_pca` | CLR em todas as categorias + PCA, mantendo `VARIANCIA_PCA` da variância | `pc_1 … pc_k` |

**Categoria de referência:** se `CATEGORIA_REFERENCIA = None`, usa `outras` quando existe (é a menos
interpretável); senão, a de menor receita.

No `clr_pca`, o PCA é ajustado numa amostra com `numpy` e aplicado em Spark como combinação linear,
sem depender de Spark ML. Os pesos (loadings) ficam nos metadados para interpretar os componentes.

In [0]:
# Célula 8 — Colinearidade
receita_modelo = cobertura.groupby("categoria_modelo").receita.sum().sort_values()
if CATEGORIA_REFERENCIA is not None:
    referencia = CATEGORIA_REFERENCIA
elif "outras" in receita_modelo.index:
    referencia = "outras"
else:
    referencia = receita_modelo.index[0]
assert referencia in categorias, f"Categoria de referência '{referencia}' não existe. Opções: {categorias}"
col_ref = f"pct_{nome_coluna(referencia)}"

COLS_CLR, loadings, var_explicada = [], None, None
if COLINEARIDADE in ("clr", "clr_pca"):
    logs = {c: F.log(F.col(c) + F.lit(PSEUDOCONTAGEM_CLR)) for c in COLS_PCT}
    media_log = reduce(add, logs.values()) / F.lit(len(COLS_PCT))
    for c in COLS_PCT:
        cesta = cesta.withColumn(c.replace("pct_", "clr_", 1), logs[c] - media_log)
    COLS_CLR = [c.replace("pct_", "clr_", 1) for c in COLS_PCT]

if COLINEARIDADE == "remover_categoria":
    COLS_MIX = [c for c in COLS_PCT if c != col_ref]
elif COLINEARIDADE == "clr":
    COLS_MIX = [c for c in COLS_CLR if c != col_ref.replace("pct_", "clr_", 1)]
else:
    frac = min(1.0, AMOSTRA_MAX / total_modelo)
    X = cesta.filter("entra_modelo = 1").select(COLS_CLR).sample(False, frac, seed=SEED).toPandas().values
    medias = X.mean(axis=0)
    _, s, vt = np.linalg.svd(X - medias, full_matrices=False)
    var_explicada = (s ** 2) / (s ** 2).sum()
    k = int(np.searchsorted(np.cumsum(var_explicada), VARIANCIA_PCA) + 1)
    loadings = pd.DataFrame(vt[:k].T, index=COLS_CLR, columns=[f"pc_{i + 1}" for i in range(k)])
    for pc in loadings.columns:
        cesta = cesta.withColumn(pc, reduce(add, [(F.col(c) - F.lit(float(m))) * F.lit(float(w))
                                                  for c, m, w in zip(COLS_CLR, medias, loadings[pc])]))
    COLS_MIX = list(loadings.columns)
    print(f"🧮 PCA: {k} componentes explicam {np.cumsum(var_explicada)[k - 1]:.1%} da variância")
    display(pd.DataFrame({"variância explicada": var_explicada, "acumulada": np.cumsum(var_explicada)}).head(k + 2))
    print("Pesos (loadings) — quanto cada categoria contribui para cada componente:")
    display(loadings.round(3))

FEATURES_MODELO = ["log_n_itens", "log_valor_total"] + COLS_MIX
print(f"🎯 Tratamento: {COLINEARIDADE} | referência: {referencia if COLINEARIDADE != 'clr_pca' else '— (PCA)'}")
print(f"   Features do modelo ({len(FEATURES_MODELO)}): {FEATURES_MODELO}")

## 9. Validação das features
Antes de gravar, confere:
1. sem nulos nem infinitos nas features do modelo (só cestas com `entra_modelo = 1`);
2. **posto (rank) da matriz** e **número de condição** da covariância, com e sem o tratamento.
   Posto menor que o nº de colunas = colinearidade perfeita. Número de condição muito alto (acima de ~10⁶)
   = matriz quase singular, instável para o GMM.

In [0]:
# Célula 9 — Validação
cesta_modelo = cesta.filter("entra_modelo = 1")
nulos = cesta_modelo.select([F.sum((F.col(c).isNull() | F.isnan(c)).cast("int")).alias(c) for c in FEATURES_MODELO]).collect()[0].asDict()
infinitos = cesta_modelo.select([F.sum(F.col(c).isin(float("inf"), float("-inf")).cast("int")).alias(c) for c in FEATURES_MODELO]).collect()[0].asDict()

frac = min(1.0, AMOSTRA_MAX / total_modelo)
amostra = cesta_modelo.select(list(dict.fromkeys(["log_n_itens", "log_valor_total"] + COLS_PCT + FEATURES_MODELO))) \
               .sample(False, frac, seed=SEED).toPandas()

def diagnostico(cols):
    z = (amostra[cols] - amostra[cols].mean()) / amostra[cols].std(ddof=0).replace(0, np.nan)
    cov = np.cov(z.fillna(0).values, rowvar=False)
    return np.linalg.matrix_rank(cov, tol=1e-8), np.linalg.cond(cov)

cols_sem = ["log_n_itens", "log_valor_total"] + COLS_PCT
r_sem, c_sem = diagnostico(cols_sem)
r_com, c_com = diagnostico(FEATURES_MODELO)

validacao = pd.DataFrame([
    {"verificação": "Sem nulos nas features do modelo", "ok": sum(nulos.values()) == 0, "detalhe": f"{sum(nulos.values())} nulos"},
    {"verificação": "Sem infinitos nas features do modelo", "ok": sum(infinitos.values()) == 0, "detalhe": f"{sum(infinitos.values())} infinitos"},
    {"verificação": "SEM tratamento: matriz de posto completo", "ok": r_sem == len(cols_sem),
     "detalhe": f"posto {r_sem} de {len(cols_sem)} | condição {c_sem:.2e}"},
    {"verificação": "COM tratamento: matriz de posto completo", "ok": r_com == len(FEATURES_MODELO),
     "detalhe": f"posto {r_com} de {len(FEATURES_MODELO)} | condição {c_com:.2e}"},
    {"verificação": "COM tratamento: condição < 1e6", "ok": c_com < 1e6, "detalhe": f"{c_com:.2e}"},
]).assign(ok=lambda d: d.ok.map({True: "✅", False: "⚠️"}))
display(validacao)
print("ℹ️ Esperado: a linha 'SEM tratamento' com ⚠️ (prova da colinearidade) e as linhas 'COM tratamento' com ✅.")

print("\nResumo das features do modelo (amostra):")
display(amostra[FEATURES_MODELO].describe().T)

## 10. Gravação e conferência
Grava a tabela de features em Delta, particionada por `ano`/`mes` (mesmo padrão da Silver), em modo
**overwrite**: a tabela é recalculada inteira a partir da Silver a cada execução. Grava também uma tabela
de metadados com os parâmetros, a categoria de referência, a lista de features do modelo e os pesos do PCA,
para o notebook de ML saber exatamente quais colunas usar.

In [0]:
# Célula 10a — Gravação
from datetime import datetime, timezone

COLS_ID       = ["id_transacao", "ano", "mes"]
COLS_PLANILHA = ["n_itens", "valor_total"] + COLS_PCT
COLS_EXTRAS   = ["n_skus", "n_categorias", "pct_itens_pesaveis", "hhi_mix", "preco_medio_item", "log_preco_medio_item"]
COLS_CONTROLE = ["tem_flag_valor", "tem_flag_fracionada", "tem_sem_categoria", "cesta_item_unico", "entra_modelo"]
colunas_finais = list(dict.fromkeys(COLS_ID + COLS_PLANILHA + COLS_EXTRAS + COLS_CONTROLE + COLS_CLR + FEATURES_MODELO))

df_features = cesta.select(*colunas_finais).withColumn("features_processed_at", F.current_timestamp())

metadados = {
    "executado_em": datetime.now(timezone.utc).isoformat(),
    "parametros": {"TRATAR_SEM_CATEGORIA": TRATAR_SEM_CATEGORIA, "LIMIAR_CATEGORIA_RARA": LIMIAR_CATEGORIA_RARA,
                   "LIMIAR_EXCLUSAO_FLAG": LIMIAR_EXCLUSAO_FLAG,
                   "TRATAR_ITEM_UNICO_A_PARTE": TRATAR_ITEM_UNICO_A_PARTE, "COLINEARIDADE": COLINEARIDADE,
                   "PSEUDOCONTAGEM_CLR": PSEUDOCONTAGEM_CLR, "VARIANCIA_PCA": VARIANCIA_PCA},
    "categoria_referencia": referencia if COLINEARIDADE != "clr_pca" else None,
    "categorias_modelo": categorias,
    "agrupadas_em_outras": agrupadas,
    "features_modelo": FEATURES_MODELO,
    "loadings_pca": loadings.round(6).to_dict() if loadings is not None else None,
    "decisao_flags": decisao_flags,
    "cestas": total_cestas,
    "cestas_modelo": total_modelo,
    "filtro_modelo": "entra_modelo = 1",
}

if GRAVAR:
    (df_features.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").options(**opcoes_adls)
        .partitionBy("ano", "mes").save(CAMINHO_FEATURES))
    (spark.createDataFrame([(metadados["executado_em"], json.dumps(metadados, ensure_ascii=False))],
                           "executado_em string, metadados_json string")
        .write.format("delta").mode("append").options(**opcoes_adls).save(CAMINHO_METADADOS))
    print(f"💾 Features gravadas em {CAMINHO_FEATURES}")
    print(f"💾 Metadados acrescentados em {CAMINHO_METADADOS}")
else:
    print("ℹ️ GRAVAR = False: nada foi gravado.")

In [0]:
# Célula 10b — Conferência pós-gravação
if GRAVAR:
    lida = spark.read.format("delta").options(**opcoes_adls).load(CAMINHO_FEATURES)
    n_lida = lida.count()
    dup = n_lida - lida.select("id_transacao").distinct().count()
    print(f"🔁 Linhas gravadas: {n_lida:,} (esperado {total_cestas:,}) {'✅' if n_lida == total_cestas else '⚠️'}")
    print(f"🔑 id_transacao duplicado: {dup:,} {'✅' if dup == 0 else '⚠️'}")
    print(f"🧱 Colunas: {len(lida.columns)}")
    print("Cestas por segmento:")
    display(lida.groupBy("entra_modelo", "cesta_item_unico").count().toPandas())
    display(lida.groupBy("ano", "mes").count().orderBy("ano", "mes").toPandas())
    display(lida.limit(5).toPandas())

## Próximos passos (notebook de ML)
1. Ler `features/cestas_gmm` e o último registro de `features/cestas_gmm_metadados`; usar `features_modelo`
   e filtrar `entra_modelo = 1`.
2. Tirar uma **amostra estratificada por `ano`/`mes`** (mesma fração em cada mês) para treinar no scikit-learn;
   ajustar o `StandardScaler` **só nessa amostra**.
3. Testar `n_components` de 2 a 8 e escolher pelo **BIC**; comparar `covariance_type` `full` e `diag`.
4. Aplicar o modelo treinado na base inteira (`entra_modelo = 1`) para dar o grupo de cada cesta.
5. Juntar o segmento "compra de item único" (`cesta_item_unico = 1`) ao resultado, como grupo definido por regra.
6. Descrever cada cluster pelas colunas originais (`n_itens`, `valor_total`, `pct_*`) e pelos extras.
7. Comparar com o K-Means da dupla 2, que usa as mesmas features da planilha.

### Decisões tomadas (com base na EDA 06.1)
| Decisão | Escolha | Evidência da EDA |
|---|---|---|
| Cestas com flag | regra: exclui se < 1% das cestas, mantém se ≥ 1% | 0,00% das cestas com flag → nada sai na prática |
| Itens sem categoria | `outras` | 0 itens sem categoria |
| Categorias raras | mantidas (18 categorias) | — |
| Colinearidade | retirar a categoria de referência | posto incompleto sem tratamento (célula 9) |
| Escala | `log1p` aqui + `StandardScaler` no ML | assimetria 1,94 → 0,08 e 2,31 → −0,54 |
| Cestas de 1 item | segmento à parte, fora do GMM | 2,67% das cestas |
| Tamanho da base | amostra estratificada por mês no scikit-learn | — |
| Extras no modelo | não entram; ficam para análise dos clusters | a alinhar com a dupla 2 |